In [1]:
!pip -q install requests sentence-transformers faiss-cpu rank_bm25 google-genai pandas

In [2]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
print("Key loaded:", bool(os.environ["GEMINI_API_KEY"]))

Key loaded: True


In [3]:
%%writefile rag_core.py
"""ADE-Assist core: data fetch (openFDA), chunking, hybrid retrieval, grounded generation."""
import os
import random
import re
import time

import numpy as np
import requests

OPENFDA_URL = "https://api.fda.gov/drug/label.json"

SECTION_FIELDS = {
    "boxed_warning": "Boxed Warning",
    "indications_and_usage": "Indications and Usage",
    "dosage_and_administration": "Dosage and Administration",
    "contraindications": "Contraindications",
    "warnings_and_cautions": "Warnings and Precautions",
    "warnings": "Warnings",
    "precautions": "Precautions",
    "adverse_reactions": "Adverse Reactions",
    "drug_interactions": "Drug Interactions",
    "overdosage": "Overdosage",
    "do_not_use": "Do Not Use",
    "ask_doctor": "Ask a Doctor Before Use",
    "stop_use": "Stop Use and Ask a Doctor",
}

REFUSAL = "I couldn't find this in the provided documents."
MODELS = ["gemini-3.5-flash", "gemini-2.5-flash", "gemini-flash-latest", "gemini-3.5-flash-lite"]

# 1. Data: fetch + chunk

def fetch_label(drug, tries=3):
    """Fetch the most complete openFDA drug label for a generic name."""
    params = {"search": f'openfda.generic_name:"{drug}"', "limit": 10}
    for _ in range(tries):
        try:
            r = requests.get(OPENFDA_URL, params=params, timeout=30)
            if r.status_code == 200:
                results = r.json().get("results", [])
                if not results:
                    return None
                return max(
                    results,
                    key=lambda x: sum(len(" ".join(x.get(f, []))) for f in SECTION_FIELDS),
                )
            if r.status_code == 404:
                return None
        except requests.RequestException:
            pass
        time.sleep(2)
    return None


def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def chunk_words(text, size=200, overlap=40):
    """Split text into overlapping word windows."""
    words = text.split()
    if not words:
        return []
    step = max(size - overlap, 1)
    chunks = []
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks


def build_chunks(labels, size=200, overlap=40, max_words=1500):
    """labels: {drug_name: openFDA label dict}. Returns a list of chunk dicts with metadata."""
    chunks = []
    for drug, label in labels.items():
        set_id = label.get("set_id", "")
        for field, section_name in SECTION_FIELDS.items():
            parts = label.get(field)
            if not parts:
                continue
            text = clean_text(" ".join(parts))
            text = " ".join(text.split()[:max_words])
            for piece in chunk_words(text, size, overlap):
                chunks.append({
                    "id": len(chunks),
                    "drug": drug,
                    "section": field,
                    "section_name": section_name,
                    "set_id": set_id,
                    "text": piece,
                })
    return chunks

# 2. Retrieval: dense (FAISS) + sparse (BM25) + Reciprocal Rank Fusion

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


class HybridRetriever:
    def __init__(self, chunks, embedder=None, embeddings=None,
                 model_name="sentence-transformers/all-MiniLM-L6-v2"):
        import faiss
        from rank_bm25 import BM25Okapi
        from sentence_transformers import SentenceTransformer

        self.chunks = chunks
        self.embedder = embedder or SentenceTransformer(model_name)

        texts = [f"{c['drug']} - {c['section_name']}: {c['text']}" for c in chunks]

        if embeddings is None:
            embeddings = self.embedder.encode(
                texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
            )
        self.embeddings = np.asarray(embeddings, dtype="float32")
        self.index = faiss.IndexFlatIP(self.embeddings.shape[1])
        self.index.add(self.embeddings)
        self.bm25 = BM25Okapi([tokenize(t) for t in texts])

    def search(self, query, k=5, mode="hybrid", pool=30):
        """mode: 'dense' | 'bm25' | 'hybrid'"""
        if mode not in ("dense", "bm25", "hybrid"):
            raise ValueError("mode must be dense, bm25 or hybrid")
        pool = min(pool, len(self.chunks))
        rankings = []
        if mode in ("dense", "hybrid"):
            q = self.embedder.encode([query], normalize_embeddings=True)
            _, idx = self.index.search(np.asarray(q, dtype="float32"), pool)
            rankings.append([int(i) for i in idx[0] if i >= 0])
        if mode in ("bm25", "hybrid"):
            scores = self.bm25.get_scores(tokenize(query))
            rankings.append([int(i) for i in np.argsort(scores)[::-1][:pool]])

        fused = {}
        for ranking in rankings:
            for rank, i in enumerate(ranking):
                fused[i] = fused.get(i, 0.0) + 1.0 / (60 + rank + 1)
        top = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]
        return [dict(self.chunks[i], score=round(s, 4)) for i, s in top]

# 3. Generation: grounded answer with citations + refusal
def build_prompt(question, hits):
    context = "\n\n".join(
        f"[{i + 1}] Drug: {h['drug']} | Section: {h['section_name']}\n{h['text']}"
        for i, h in enumerate(hits)
    )
    return f"""You are ADE-Assist, a drug-safety information assistant.

Rules:
1. Answer ONLY using the CONTEXT below. Do not use outside knowledge.
2. Cite the supporting passages like [1], [2] right after each claim.
3. If the context does not contain the answer, reply exactly: "{REFUSAL}"
4. Be concise (at most 6 sentences). Do not give personal medical advice, personal dosing, or diagnoses.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""


def generate(prompt, api_key=None, models=None, retries=4):
    """Call Gemini with exponential backoff; skip to the next model on permanent errors."""
    from google import genai

    key = api_key or os.environ.get("GEMINI_API_KEY")
    if not key:
        raise RuntimeError("GEMINI_API_KEY is not set")
    client = genai.Client(api_key=key)
    errors = {}
    for model in (models or MODELS):
        for attempt in range(retries):
            try:
                resp = client.models.generate_content(model=model, contents=prompt)
                return (resp.text or "").strip()
            except Exception as e:
                errors[model] = str(e)[:120]
                msg = str(e)
                # permanent errors: retrying this model is pointless, go to the next one
                if any(w in msg for w in ("404", "NOT_FOUND", "403", "PERMISSION_DENIED",
                                          "INVALID_ARGUMENT", "API key")):
                    break
                # transient (503 UNAVAILABLE, 429, timeouts): back off 2s, 4s, 8s... + jitter
                time.sleep(min(2 ** (attempt + 1), 30) + random.random())
    details = "; ".join(f"{m}: {err}" for m, err in errors.items())
    raise RuntimeError(f"All Gemini models failed. Per-model errors -> {details}")


def answer_question(question, retriever, k=5, mode="hybrid", api_key=None):
    hits = retriever.search(question, k=k, mode=mode)
    text = generate(build_prompt(question, hits), api_key=api_key)
    return {"answer": text, "sources": hits, "refused": REFUSAL.lower() in text.lower()}

# 4. Evaluation helpers

def evaluate_retrieval(retriever, eval_set, k=5, mode="hybrid"):
    """eval_set items: {"q": str, "drug": str, "sections": [section_field, ...]}"""
    drug_hits = section_hits = 0
    for ex in eval_set:
        hits = retriever.search(ex["q"], k=k, mode=mode)
        right_drug = [h for h in hits if h["drug"] == ex["drug"]]
        if right_drug:
            drug_hits += 1
        if any(h["section"] in ex["sections"] for h in right_drug):
            section_hits += 1
    n = len(eval_set)
    return {
        "mode": mode,
        "k": k,
        "drug_hit@k": round(drug_hits / n, 3),
        "drug+section_hit@k": round(section_hits / n, 3),
    }

Writing rag_core.py


In [4]:
import json, time
from rag_core import fetch_label, build_chunks

DRUGS = [
    "ibuprofen", "acetaminophen", "aspirin", "naproxen", "diclofenac", "metformin",
    "atorvastatin", "simvastatin", "lisinopril", "losartan", "amlodipine", "metoprolol",
    "furosemide", "warfarin", "clopidogrel", "omeprazole", "amoxicillin", "azithromycin",
    "ciprofloxacin", "sertraline", "fluoxetine", "levothyroxine", "gabapentin", "prednisone",
    "tramadol", "montelukast", "cetirizine", "loratadine", "ondansetron",
]

labels = {}
for d in DRUGS:
    lab = fetch_label(d)
    if lab:
        labels[d] = lab
    else:
        print("No label found for:", d)
    time.sleep(0.3)

print(f"Fetched {len(labels)} labels")
json.dump(labels, open("labels.json", "w"))

Fetched 29 labels


In [5]:
chunks = build_chunks(labels, size=200, overlap=40)
print("Total chunks:", len(chunks))

import pandas as pd
df = pd.DataFrame(chunks)
print(df.groupby("drug").size().sort_values(ascending=False).head(10))
print(df.groupby("section_name").size().sort_values(ascending=False))
chunks[0]

Total chunks: 931
drug
sertraline       61
tramadol         52
fluoxetine       51
ciprofloxacin    45
prednisone       43
lisinopril       42
gabapentin       40
simvastatin      38
naproxen         38
atorvastatin     37
dtype: int64
section_name
Adverse Reactions            177
Dosage and Administration    166
Warnings and Precautions     149
Drug Interactions            148
Indications and Usage         77
Precautions                   65
Warnings                      47
Overdosage                    37
Contraindications             32
Boxed Warning                 24
Ask a Doctor Before Use        3
Do Not Use                     3
Stop Use and Ask a Doctor      3
dtype: int64


{'id': 0,
 'drug': 'ibuprofen',
 'section': 'boxed_warning',
 'section_name': 'Boxed Warning',
 'set_id': '0170876a-d8cd-4a2d-8397-db2acdf72ffb',
 'text': 'Cardiovascular Thrombotic Events Nonsteroidal anti-inflammatory drugs (NSAIDs) cause an increased risk of serious cardiovascular thrombotic events, including myocardial infarction and stroke, which can be fatal. This risk may occur early in treatment and may increase with duration of use. [See WARNINGS and PRECAUTIONS ]. Ibuprofen tablets are contraindicated in the setting of coronary artery bypass graft (CABG) surgery [see CONTRAINDICATIONS and WARNINGS ]. Gastrointestinal Risk NSAIDS cause an increased risk of serious gastrointestinal adverse events including bleeding, ulceration, and perforation of the stomach or intestines, which can be fatal. These events can occur at any time during use and without warning symptoms. Elderly patients are at greater risk for serious gastrointestinal events. [See WARNINGS ].'}

In [6]:
from rag_core import HybridRetriever
retriever = HybridRetriever(chunks)

for hit in retriever.search("Which antibiotic can cause tendon rupture?", k=3):
    print(hit["drug"], "|", hit["section_name"], "| score", hit["score"])
    print(hit["text"][:200], "...\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

ciprofloxacin | Warnings and Precautions | score 0.0328
tendinitis, tendon rupture, arthralgia, myalgia, peripheral neuropathy, and central nervous system effects (hallucinations, anxiety, depression, insomnia, severe headaches, and confusion). These react ...

ciprofloxacin | Warnings and Precautions | score 0.0323
within hours or days of starting ciprofloxacin, or as long as several months after completion of fluoroquinolone therapy. Tendinitis and tendon rupture can occur bilaterally. The risk of developing fl ...

prednisone | Precautions | score 0.0317
corticosteroids, thereby increasing their effect. Fluoroquinolones Post-marketing surveillance reports indicate that the risk of tendon rupture may be increased in patients receiving concomitant fluor ...



In [7]:
from rag_core import answer_question

res = answer_question("Does sertraline carry a warning about suicidal thoughts in young people?", retriever)
print(res["answer"])
print()
for i, h in enumerate(res["sources"], 1):
    print(f"[{i}] {h['drug']} | {h['section_name']}")

Yes, sertraline carries a boxed warning regarding suicidal thoughts and behaviors in young people [1]. In short-term studies, antidepressants increased the risk of suicidal thinking and behavior (suicidality) compared to placebo in children, adolescents, and young adults (ages 18 to 24) with major depressive disorder and other psychiatric disorders [1], [3]. Anyone considering the use of sertraline hydrochloride in a child, adolescent, or young adult must balance this risk with the clinical need [1]. Patients of all ages started on therapy should be monitored appropriately and observed closely for clinical worsening, suicidality, or unusual changes in behavior [1], [4].

[1] sertraline | Boxed Warning
[2] fluoxetine | Boxed Warning
[3] sertraline | Warnings
[4] sertraline | Precautions
[5] fluoxetine | Boxed Warning


In [8]:
EVAL_SET = [
    {"q": "Does the NSAID diclofenac carry a warning about heart attack and stroke?", "drug": "diclofenac",
     "sections": ["boxed_warning", "warnings_and_cautions", "warnings"]},
    {"q": "Which blood thinner has a boxed warning about major or fatal bleeding?", "drug": "warfarin",
     "sections": ["boxed_warning", "warnings_and_cautions"]},
    {"q": "Can the diabetes drug metformin cause lactic acidosis?", "drug": "metformin",
     "sections": ["boxed_warning", "warnings_and_cautions", "warnings"]},
    {"q": "Which blood pressure drug is dangerous in pregnancy because of fetal toxicity?", "drug": "lisinopril",
     "sections": ["boxed_warning", "warnings_and_cautions", "contraindications"]},
    {"q": "Which antiplatelet drug works less well in people who are poor CYP2C19 metabolizers?", "drug": "clopidogrel",
     "sections": ["boxed_warning", "warnings_and_cautions", "drug_interactions"]},
    {"q": "What are common side effects of the cholesterol drug atorvastatin?", "drug": "atorvastatin",
     "sections": ["adverse_reactions"]},
    {"q": "What is the usual starting dose of the thyroid hormone levothyroxine?", "drug": "levothyroxine",
     "sections": ["dosage_and_administration"]},
    {"q": "What is the maximum daily dose of acetaminophen for adults?", "drug": "acetaminophen",
     "sections": ["dosage_and_administration", "warnings", "do_not_use", "ask_doctor", "stop_use"]},
    {"q": "Does the antidepressant sertraline carry a warning about suicidal thoughts in young people?", "drug": "sertraline",
     "sections": ["boxed_warning", "warnings_and_cautions"]},
    {"q": "Which antibiotic can cause tendon rupture?", "drug": "ciprofloxacin",
     "sections": ["boxed_warning", "warnings_and_cautions"]},
    {"q": "What risks come from taking tramadol together with sedating drugs?", "drug": "tramadol",
     "sections": ["boxed_warning", "drug_interactions", "warnings_and_cautions"]},
    {"q": "Is montelukast linked to neuropsychiatric events?", "drug": "montelukast",
     "sections": ["boxed_warning", "warnings_and_cautions"]},
    # TODO: add 15+ more of your own questions here
]
print(len(EVAL_SET), "questions")

12 questions


In [9]:
from rag_core import evaluate_retrieval
import pandas as pd

rows = []
for mode in ["dense", "bm25", "hybrid"]:
    for k in [3, 5]:
        rows.append(evaluate_retrieval(retriever, EVAL_SET, k=k, mode=mode))
retrieval_results = pd.DataFrame(rows)
retrieval_results

,mode,k,drug_hit@k,drug+section_hit@k
0,dense,3,1.0,0.917
1,dense,5,1.0,0.917
2,bm25,3,1.0,0.750
3,bm25,5,1.0,0.750
4,hybrid,3,1.0,0.917
5,hybrid,5,1.0,0.917


In [10]:
rows = []
for size, overlap in [(120, 20), (200, 40), (350, 60)]:
    ch = build_chunks(labels, size=size, overlap=overlap)
    r = HybridRetriever(ch, embedder=retriever.embedder)
    res = evaluate_retrieval(r, EVAL_SET, k=5, mode="hybrid")
    res.update({"chunk_words": size, "n_chunks": len(ch)})
    rows.append(res)
chunk_results = pd.DataFrame(rows)
chunk_results

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

,mode,k,drug_hit@k,drug+section_hit@k,chunk_words,n_chunks
0,hybrid,5,0.917,0.833,120,1426
1,hybrid,5,1.000,0.917,200,931
2,hybrid,5,1.000,0.833,350,537


In [11]:
import time
from rag_core import REFUSAL

OUT_OF_SCOPE = [
    "Who won the cricket world cup in 2011?",
    "What is the capital of France?",
    "How do I invest in the stock market?",
    "Write me a poem about the monsoon.",
]
refused = 0
tested = 0
for q in OUT_OF_SCOPE:
    try:
        res = answer_question(q, retriever)
    except Exception as e:
        print("SKIPPED (Gemini error) |", q, "->", str(e)[:150])
        time.sleep(10)
        continue
    tested += 1
    ok = res["refused"]
    refused += ok
    print("REFUSED" if ok else "ANSWERED", "|", q, "->", res["answer"][:80])
    time.sleep(8)
print(f"Refusal rate: {refused}/{tested} answered questions (out of {len(OUT_OF_SCOPE)})")


REFUSED | Who won the cricket world cup in 2011? -> I couldn't find this in the provided documents.
REFUSED | What is the capital of France? -> I couldn't find this in the provided documents.
REFUSED | How do I invest in the stock market? -> I couldn't find this in the provided documents.
REFUSED | Write me a poem about the monsoon. -> I couldn't find this in the provided documents.
Refusal rate: 4/4 answered questions (out of 4)


In [12]:
import time, os, rag_core
from google import genai

def generate_patient(prompt, api_key=None, models=None, retries=6):
    key = api_key or os.environ.get("GEMINI_API_KEY")
    client = genai.Client(api_key=key)
    last_err = None
    for model in (models or rag_core.MODELS):
        for attempt in range(retries):
            try:
                resp = client.models.generate_content(model=model, contents=prompt)
                return (resp.text or "").strip()
            except Exception as e:
                last_err = e
                msg = str(e)
                if any(w in msg for w in ("404", "NOT_FOUND", "403", "PERMISSION_DENIED", "API key")):
                    break
                time.sleep(min(2 ** (attempt + 1), 30))
    raise RuntimeError(f"All Gemini models failed. Last error: {last_err}")

rag_core.generate = generate_patient

for ex in EVAL_SET[:5]:
    print("Q:", ex["q"])
    try:
        res = rag_core.answer_question(ex["q"], retriever)
        print("A:", res["answer"])
    except Exception as e:
        print("FAILED (Gemini busy):", str(e)[:150])
    print("-" * 80)
    time.sleep(10)

Q: Does the NSAID diclofenac carry a warning about heart attack and stroke?
A: Yes, nonsteroidal anti-inflammatory drugs (NSAIDs) like diclofenac carry an increased risk of serious cardiovascular thrombotic events, including myocardial infarction (heart attack) and stroke, which can be fatal [4], [5]. This risk can occur early in treatment and may increase with the duration of use [4], [5]. Additionally, NSAIDs are contraindicated in the setting of coronary artery bypass graft (CABG) surgery [1], [2], [4].
--------------------------------------------------------------------------------
Q: Which blood thinner has a boxed warning about major or fatal bleeding?
A: Warfarin has a boxed warning stating that warfarin sodium can cause major or fatal bleeding [1].
--------------------------------------------------------------------------------
Q: Can the diabetes drug metformin cause lactic acidosis?
A: Yes, postmarketing cases of metformin-associated lactic acidosis, including fatal cases, ha

In [13]:
import numpy as np
json.dump(chunks, open("chunks.json", "w"))
np.save("embeddings.npy", retriever.embeddings)
retrieval_results.to_csv("retrieval_results.csv", index=False)
chunk_results.to_csv("chunk_size_results.csv", index=False)
print("Saved chunks.json, embeddings.npy and result CSVs")

Saved chunks.json, embeddings.npy and result CSVs


In [14]:
APP = 'import json\nimport os\n\nimport numpy as np\nimport streamlit as st\n\nfrom rag_core import HybridRetriever, answer_question\n\nst.set_page_config(page_title="ADE-Assist", page_icon="💊", layout="wide")\n\n\n@st.cache_resource(show_spinner="Loading drug-label index...")\ndef load_retriever():\n    with open("chunks.json", encoding="utf-8") as f:\n        chunks = json.load(f)\n    embeddings = np.load("embeddings.npy")\n    return HybridRetriever(chunks, embeddings=embeddings)\n\n\ndef get_api_key():\n    try:\n        return st.secrets["GEMINI_API_KEY"]\n    except Exception:\n        return os.environ.get("GEMINI_API_KEY")\n\n\nst.title("💊 ADE-Assist: Drug Safety RAG Assistant")\nst.caption(\n    "Answers come only from FDA drug labels (openFDA) and always cite their source. "\n    "Educational use only. This is not medical advice."\n)\n\nretriever = load_retriever()\napi_key = get_api_key()\n\nwith st.sidebar:\n    st.header("Settings")\n    mode = st.selectbox("Retrieval mode", ["hybrid", "dense", "bm25"], index=0)\n    k = st.slider("Passages to retrieve (k)", 2, 10, 5)\n    if not api_key:\n        api_key = st.text_input("Gemini API key", type="password")\n    st.markdown("**Try:**")\n    st.markdown("- Which antibiotic can cause tendon rupture?\\n"\n                "- Does sertraline carry a suicidality warning?\\n"\n                "- Can metformin cause lactic acidosis?")\n\nquestion = st.text_input("Ask a drug-safety question")\n\nif st.button("Ask") and question.strip():\n    if not api_key:\n        st.error("Please provide a Gemini API key in the sidebar.")\n    else:\n        with st.spinner("Searching labels and generating answer..."):\n            try:\n                result = answer_question(question, retriever, k=k, mode=mode, api_key=api_key)\n            except Exception as e:\n                st.error(f"Something went wrong: {e}")\n            else:\n                if result["refused"]:\n                    st.warning(result["answer"])\n                else:\n                    st.markdown(result["answer"])\n                with st.expander("Sources used"):\n                    for i, h in enumerate(result["sources"], 1):\n                        st.markdown(f"**[{i}] {h[\'drug\'].title()} | {h[\'section_name\']}** "\n                                    f"(score {h[\'score\']})")\n                        st.write(h["text"])\n'
REQ = 'streamlit\nnumpy\nrequests\nsentence-transformers\nfaiss-cpu\nrank_bm25\ngoogle-genai\n'
README = '# ADE-Assist: Drug Safety RAG Assistant\n\nA retrieval-augmented question-answering system over FDA drug labels (openFDA). It returns short, **source-cited** answers and refuses when the documents don\'t contain the answer.\n\n> Educational project. Not medical advice.\n\n**Live demo:** `<add your link>`\n\n## Problem\nDrug labels are long and hard to scan. ADE-Assist lets you ask a natural-language question (for example, "Which antibiotic can cause tendon rupture?") and get a grounded answer with the exact passages it used.\n\n## Architecture\n```\nopenFDA labels -> clean -> chunk (200 words, 40 overlap, drug+section metadata)\n      -> embeddings (MiniLM) -> FAISS   \\\n      -> BM25 index                      > Reciprocal Rank Fusion -> top-k passages\n                                         /\nquestion ------------------------------/\n      -> Gemini (answer ONLY from passages, cite [n], else refuse) -> answer + sources\n```\n\n## Tech stack\nPython, openFDA API, sentence-transformers, FAISS, rank_bm25, Gemini API, Streamlit.\n\n## Evaluation\nHand-written question set: `<N>` questions with a known drug and label section.\n\n| Retrieval | drug hit@5 | drug+section hit@5 |\n|---|---|---|\n| Dense only | `<fill>` | `<fill>` |\n| BM25 only | `<fill>` | `<fill>` |\n| **Hybrid (RRF)** | `<fill>` | `<fill>` |\n\nChunk-size ablation (words): `<fill 120 / 200 / 350 results>`\nOut-of-scope refusal test: `<x>/<y>` questions correctly refused.\n\n## Run locally\n```bash\npip install -r requirements.txt\nexport GEMINI_API_KEY=your_key\nstreamlit run app.py\n```\nRebuild the index by running the Colab notebook, which produces `chunks.json` and `embeddings.npy`.\n\n## Limitations\n- Covers about 30 common drugs, not the whole FDA database.\n- Label text can be outdated and varies between manufacturers.\n- Evaluation set is small and hand-written.\n- Generation faithfulness was only spot-checked, not measured automatically.\n\n## Future work\nLink to ADEGuard (predict adverse-event severity, then explain it from labels), add a reranker, and automatic faithfulness scoring.\n'
open("app.py", "w", encoding="utf-8").write(APP)
open("requirements.txt", "w", encoding="utf-8").write(REQ)
open("README.md", "w", encoding="utf-8").write(README)

from google.colab import files
for f in ["rag_core.py", "app.py", "requirements.txt", "README.md", "chunks.json",
          "embeddings.npy", "retrieval_results.csv", "chunk_size_results.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (6 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
NEW_QUESTIONS = [
    {"q": "Which blood thinner needs regular INR monitoring?", "drug": "warfarin",
     "sections": ["boxed_warning", "dosage_and_administration", "warnings_and_cautions", "warnings"]},
    {"q": "Which thyroid medicine must not be used for obesity or weight loss?", "drug": "levothyroxine",
     "sections": ["boxed_warning", "warnings_and_cautions", "warnings", "indications_and_usage"]},
    {"q": "Which common painkiller can damage the liver in an overdose?", "drug": "acetaminophen",
     "sections": ["warnings", "boxed_warning", "overdosage", "ask_doctor", "stop_use", "do_not_use"]},
    {"q": "Which macrolide antibiotic can prolong the QT interval?", "drug": "azithromycin",
     "sections": ["warnings_and_cautions", "warnings", "precautions"]},
    {"q": "Which pain medicine can cause seizures and serotonin syndrome, and is dangerous for children who metabolize it very fast?", "drug": "tramadol",
     "sections": ["boxed_warning", "warnings_and_cautions", "warnings"]},
    {"q": "Which proton pump inhibitor is used for acid reflux and stomach ulcers?", "drug": "omeprazole",
     "sections": ["indications_and_usage"]},
    {"q": "Which anticonvulsant is also used for nerve pain after shingles?", "drug": "gabapentin",
     "sections": ["indications_and_usage"]},
    {"q": "Which corticosteroid can suppress the adrenal glands with long-term use?", "drug": "prednisone",
     "sections": ["warnings", "precautions", "warnings_and_cautions"]},
    {"q": "Which loop diuretic can cause profound loss of water and electrolytes if given in excess?", "drug": "furosemide",
     "sections": ["boxed_warning", "warnings", "warnings_and_cautions"]},
    {"q": "Which beta-blocker warns against stopping suddenly because of chest pain or heart attack risk?", "drug": "metoprolol",
     "sections": ["boxed_warning", "warnings", "warnings_and_cautions"]},
    {"q": "Which angiotensin receptor blocker carries a fetal toxicity warning?", "drug": "losartan",
     "sections": ["boxed_warning", "warnings_and_cautions", "warnings"]},
    {"q": "Which calcium channel blocker commonly causes swelling in the ankles and feet?", "drug": "amlodipine",
     "sections": ["adverse_reactions"]},
    {"q": "Which anti-nausea drug is a 5-HT3 blocker that can prolong the QT interval?", "drug": "ondansetron",
     "sections": ["warnings_and_cautions", "warnings", "precautions", "adverse_reactions"]},


    {"q": "What is the recommended adult dose of amoxicillin for mild to moderate infections?", "drug": "amoxicillin",
     "sections": ["dosage_and_administration"]},
    {"q": "How should metformin be handled in patients with kidney problems?", "drug": "metformin",
     "sections": ["dosage_and_administration", "contraindications", "warnings_and_cautions", "warnings", "precautions"]},
    {"q": "Is it safe to take warfarin together with aspirin or NSAIDs?", "drug": "warfarin",
     "sections": ["drug_interactions", "warnings_and_cautions", "boxed_warning", "warnings"]},
    {"q": "Which medicine must not be combined with ciprofloxacin because of a dangerous interaction?", "drug": "ciprofloxacin",
     "sections": ["contraindications", "drug_interactions", "warnings_and_cautions"]},
    {"q": "Can fluoxetine be taken together with thioridazine?", "drug": "fluoxetine",
     "sections": ["contraindications", "drug_interactions", "warnings_and_cautions", "warnings"]},
    {"q": "What are the signs of an overdose of gabapentin?", "drug": "gabapentin",
     "sections": ["overdosage"]},
    {"q": "What muscle or liver problems can happen with simvastatin?", "drug": "simvastatin",
     "sections": ["warnings_and_cautions", "warnings", "precautions", "adverse_reactions"]},
]

EVAL_SET = EVAL_SET[:12] + NEW_QUESTIONS   # safe to re-run: never duplicates
print(len(EVAL_SET), "questions in total")

32 questions in total


In [16]:
from collections import defaultdict

have = defaultdict(set)
for c in chunks:
    have[c["drug"]].add(c["section"])

problems = 0
for ex in EVAL_SET:
    if ex["drug"] not in have:
        problems += 1
        print("DRUG MISSING:", ex["drug"], "|", ex["q"])
    elif not any(s in have[ex["drug"]] for s in ex["sections"]):
        problems += 1
        print("CHECK THIS:", ex["q"])
        print("   drug:", ex["drug"], "| sections you listed:", ex["sections"])
        print("   sections that really exist:", sorted(have[ex["drug"]]))
print("Questions to fix:", problems)

Questions to fix: 0


In [17]:
from rag_core import evaluate_retrieval
import pandas as pd

rows = []
for mode in ["dense", "bm25", "hybrid"]:
    for k in [3, 5]:
        rows.append(evaluate_retrieval(retriever, EVAL_SET, k=k, mode=mode))
retrieval_results = pd.DataFrame(rows)
print("Questions:", len(EVAL_SET))
retrieval_results

Questions: 32


,mode,k,drug_hit@k,drug+section_hit@k
0,dense,3,0.938,0.875
1,dense,5,0.938,0.906
2,bm25,3,0.906,0.750
3,bm25,5,0.906,0.781
4,hybrid,3,0.906,0.844
5,hybrid,5,0.938,0.875


In [18]:
def show_misses(retriever, eval_set, k=5, mode="hybrid"):
    misses = 0
    for ex in eval_set:
        hits = retriever.search(ex["q"], k=k, mode=mode)
        right = [h for h in hits if h["drug"] == ex["drug"] and h["section"] in ex["sections"]]
        if not right:
            misses += 1
            print("MISS:", ex["q"])
            print("   expected:", ex["drug"], ex["sections"])
            for h in hits:
                print("   got:", h["drug"], "|", h["section"])
    print(f"\n{misses} misses out of {len(eval_set)} ({mode}, k={k})")

show_misses(retriever, EVAL_SET, k=5, mode="hybrid")

MISS: Can the diabetes drug metformin cause lactic acidosis?
   expected: metformin ['boxed_warning', 'warnings_and_cautions', 'warnings']
   got: metformin | precautions
   got: metformin | precautions
   got: metformin | drug_interactions
   got: metformin | precautions
   got: metformin | precautions
MISS: Which anticonvulsant is also used for nerve pain after shingles?
   expected: gabapentin ['indications_and_usage']
   got: diclofenac | indications_and_usage
   got: gabapentin | dosage_and_administration
   got: fluoxetine | drug_interactions
   got: gabapentin | adverse_reactions
   got: fluoxetine | drug_interactions
MISS: Which calcium channel blocker commonly causes swelling in the ankles and feet?
   expected: amlodipine ['adverse_reactions']
   got: losartan | warnings_and_cautions
   got: metoprolol | drug_interactions
   got: lisinopril | precautions
   got: simvastatin | drug_interactions
   got: warfarin | warnings_and_cautions
MISS: Is it safe to take warfarin together

In [19]:
# Re-run chunk-size test + save CSVs on the full 32-question set
print("Questions:", len(EVAL_SET))   # must print 32

rows = []
for size, overlap in [(120, 20), (200, 40), (350, 60)]:
    ch = build_chunks(labels, size=size, overlap=overlap)
    r = HybridRetriever(ch, embedder=retriever.embedder)
    res = evaluate_retrieval(r, EVAL_SET, k=5, mode="hybrid")
    res.update({"chunk_words": size, "n_chunks": len(ch)})
    rows.append(res)
chunk_results = pd.DataFrame(rows)
display(chunk_results)

retrieval_results.to_csv("retrieval_results.csv", index=False)
chunk_results.to_csv("chunk_size_results.csv", index=False)
print("Saved updated CSVs")


Questions: 32


Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

,mode,k,drug_hit@k,drug+section_hit@k,chunk_words,n_chunks
0,hybrid,5,0.844,0.781,120,1426
1,hybrid,5,0.938,0.875,200,931
2,hybrid,5,0.938,0.875,350,537


Saved updated CSVs
